# 신규 데이터 EDA

**⚠️ 원본 데이터 절대 수정 금지 — 읽기 전용 분석**

- `Membership_train.csv`
- `Movies.csv`
- `Views_train.csv`
- `mapping.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

RAW = Path('..') / 'data/01_raw'

mem   = pd.read_csv(RAW / 'Membership_train.csv', encoding='utf-8-sig')
movie = pd.read_csv(RAW / 'Movies.csv',           encoding='utf-8-sig')
views = pd.read_csv(RAW / 'Views_train.csv',      encoding='utf-8-sig')
mapp  = pd.read_csv(RAW / 'mapping.csv',          encoding='utf-8-sig')

print('Membership_train:', mem.shape)
print('Movies:          ', movie.shape)
print('Views_train:     ', views.shape)
print('mapping:         ', mapp.shape)

## 1. Membership_train

In [ ]:
print('=== 기본 정보 ===')
print(mem.dtypes)
print()
print('=== 결측치 ===')
print(mem.isnull().sum())
print()
print('=== 샘플 ===')
mem.head()

In [ ]:
# 타겟 분포
churn_rate = (mem['Repurchase'] == 'N').mean()
print(f'이탈률 (Repurchase=N): {churn_rate*100:.1f}%')
print(mem['Repurchase'].value_counts())

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Membership_train 주요 변수 분포', fontsize=13)

# 이탈률
val = mem['Repurchase'].value_counts()
axes[0,0].bar(['재구매(Y)','이탈(N)'], [val.get('Y',0), val.get('N',0)],
               color=['#3498db','#e74c3c'])
axes[0,0].set_title('재구매 여부')

# 가격 분포
axes[0,1].hist(mem['pgamount'].dropna(), bins=30, color='steelblue', edgecolor='white')
axes[0,1].set_title('결제 금액 분포')

# 연령대
mem['agegroup'].value_counts().sort_index().plot(kind='bar', ax=axes[0,2], color='steelblue')
axes[0,2].set_title('연령대 분포')
axes[0,2].tick_params(axis='x', rotation=0)

# 성별
mem['gender'].value_counts().plot(kind='bar', ax=axes[1,0], color=['#e74c3c','#3498db','gray'])
axes[1,0].set_title('성별 분포')
axes[1,0].tick_params(axis='x', rotation=0)

# 결제 기기
mem['devicetypeid'].value_counts().plot(kind='bar', ax=axes[1,1], color='steelblue')
axes[1,1].set_title('결제 기기')

# 가입 시간
mem['registerhour'].value_counts().sort_index().plot(kind='bar', ax=axes[1,2], color='steelblue')
axes[1,2].set_title('가입 시간대')

plt.tight_layout()
plt.show()

In [ ]:
# 이탈률 × 주요 변수
mem['is_churn'] = (mem['Repurchase'] == 'N').astype(int)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('주요 변수별 이탈률', fontsize=13)

# 성별별 이탈률
mem.groupby('gender')['is_churn'].mean().plot(kind='bar', ax=axes[0], color='#e74c3c')
axes[0].set_title('성별별 이탈률')
axes[0].set_ylabel('이탈률')
axes[0].tick_params(axis='x', rotation=0)

# 연령대별 이탈률
mem.groupby('agegroup')['is_churn'].mean().sort_index().plot(kind='bar', ax=axes[1], color='#e74c3c')
axes[1].set_title('연령대별 이탈률')
axes[1].tick_params(axis='x', rotation=0)

# 프로모션별 이탈률
mem.groupby('promo_100')['is_churn'].mean().plot(kind='bar', ax=axes[2], color='#e74c3c')
axes[2].set_title('프로모션(100원)별 이탈률')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 2. Movies (신규 Category 컬럼)

In [ ]:
print(movie.dtypes)
print()
print('Category 유니크:', movie['Category'].nunique(), '개')
print(movie['Category'].value_counts().head(15))
movie.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 장르 분포
movie['Category'].value_counts().head(12).plot(
    kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('장르(Category) 분포 Top12')

# 출시 월 분포
movie['RELEASE_MONTH'].astype(str).str[:4].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('출시 연도 분포')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Views_train

In [ ]:
print('=== 기본 정보 ===')
print(views.dtypes)
print()
print('결측치:', views.isnull().sum().sum())
print('유저 수:', views['USER_ID'].nunique())
print('영화 수:', views['MOVIE_ID'].nunique())
print()
print('=== 시청 시간(DURATION) ===')
print(views['DURATION'].describe())
views.head()

In [ ]:
# 유저별 시청 집계
user_stats = views.groupby('USER_ID').agg(
    총시청횟수=('DURATION','count'),
    총시청시간=('DURATION','sum'),
    고유영화수=('MOVIE_ID','nunique'),
    시청일수=('WATCH_DAY','nunique'),
).reset_index()

print('유저별 시청 통계')
print(user_stats.describe().round(1))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['총시청횟수','총시청시간','시청일수']):
    ax.hist(user_stats[col].clip(upper=user_stats[col].quantile(0.95)), bins=40,
            color='steelblue', edgecolor='white')
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 4. 이탈 여부 × 시청 행동 비교

In [ ]:
# mapping으로 연결
mem_uid = mem[['uno','is_churn']].rename(columns={'uno':'uid'})
mapp_merge = mapp.merge(mem_uid, on='uid', how='inner')
views_churn = views.merge(mapp_merge[['USER_ID','is_churn']], on='USER_ID', how='inner')

user_churn = views_churn.groupby(['USER_ID','is_churn']).agg(
    총시청횟수=('DURATION','count'),
    총시청시간=('DURATION','sum'),
    고유영화수=('MOVIE_ID','nunique'),
    시청일수=('WATCH_DAY','nunique'),
).reset_index()

print('이탈 vs 재구매 시청 행동 비교')
print(user_churn.groupby('is_churn')[['총시청횟수','총시청시간','고유영화수','시청일수']].mean().round(2))

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, ['총시청횟수','총시청시간','고유영화수','시청일수']):
    sns.boxplot(data=user_churn, x='is_churn', y=col, ax=ax,
                palette=['#3498db','#e74c3c'],
                showfliers=False)
    ax.set_xticklabels(['재구매','이탈'])
    ax.set_title(col)
plt.suptitle('이탈 vs 재구매 유저 시청 행동', fontsize=13)
plt.tight_layout()
plt.show()

## 5. 장르별 이탈률

In [ ]:
# 시청 이력 + 영화 장르 + 이탈 여부 연결
views_movie = views_churn.merge(movie[['MOVIE_ID','Category']], on='MOVIE_ID', how='left')

genre_churn = views_movie.groupby('Category')['is_churn'].mean().sort_values(ascending=False)
genre_count = views_movie.groupby('Category')['is_churn'].count()

# 시청 수 100 이상인 장르만
genre_churn = genre_churn[genre_count >= 100]

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#e74c3c' if v > genre_churn.mean() else '#3498db' for v in genre_churn.values]
genre_churn.plot(kind='bar', ax=ax, color=colors)
ax.axhline(genre_churn.mean(), linestyle='--', color='gray', label=f'평균 이탈률')
ax.set_title('장르별 이탈률')
ax.set_ylabel('이탈률')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

print('장르별 이탈률 (높은 순):')
print(genre_churn.round(3))